# Survey Dashboard

**Forensic question:** Across every qualifying author in the newsroom, where does the AI-adoption signal concentrate?

**Phase context:** Phase 12 §6a — newsroom-wide survey render. Loads the latest `data/survey/run_*/survey_results.json` and produces a ranked table, score distribution, and change-point timeline.

**Input artifacts:**
- `data/survey/run_<run_id>/survey_results.json` (most recent by mtime)
- `data/preregistration/preregistration_lock.json` (optional; cited in methodology)

**Output artifacts:** None (presentation-only). This notebook is safe to run before any survey has executed — it will display a friendly placeholder and skip the plots.

**Reproduction:** `uv run forensics survey` then re-render with `quarto render notebooks/10_survey_dashboard.ipynb`.

In [ ]:
# | echo: false
from __future__ import annotations

import json
from pathlib import Path

import polars as pl

from forensics.config import get_project_root

ROOT = get_project_root()
SURVEY_DIR = ROOT / "data" / "survey"


def _latest_survey_results() -> Path | None:
    if not SURVEY_DIR.is_dir():
        return None
    candidates = sorted(
        SURVEY_DIR.glob("run_*/survey_results.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


results_path = _latest_survey_results()
if results_path is None:
    print("No survey results found under data/survey/. Run `uv run forensics survey` first.")
    payload = None
else:
    payload = json.loads(results_path.read_text(encoding="utf-8"))
    print(f"Loaded: {results_path.relative_to(ROOT)}")
    print(f"run_id={payload.get('run_id')} config_hash={payload.get('config_hash')}")

## Top-10 authors by composite score

Ranked by composite AI-adoption score, with signal-strength tier as a badge. Tiers follow §1d thresholds — `STRONG` requires composite ≥ 0.7 and converging effect sizes; `NONE` is explicit "no signal above threshold."

In [ ]:
if payload is None:
    print("(no survey data — table skipped)")
    top10 = None
else:
    rankings = payload.get("rankings", [])
    if not rankings:
        print("(survey has no rankings yet)")
        top10 = None
    else:
        df = pl.from_dicts(rankings)
        keep_cols = [
            c
            for c in (
                "rank",
                "author_slug",
                "author_name",
                "composite_score",
                "signal_strength",
                "total_articles",
                "convergence_windows",
            )
            if c in df.columns
        ]
        top10 = df.select(keep_cols).head(10)
        print(top10)

## Composite score distribution

Histogram of composite scores across all processed authors. The natural-control cohort (authors automatically tagged `NONE` below the control threshold) is overlaid in a distinct tier so the separation between control and flagged populations is visible.

In [ ]:
if payload is None:
    print("(no survey data — distribution skipped)")
else:
    import plotly.graph_objects as go  # noqa: I001

    rankings = payload.get("rankings", [])
    control_set = set(payload.get("natural_controls", []))
    if not rankings:
        print("(no rankings yet)")
    else:
        df = pl.from_dicts(rankings).drop_nulls(subset=["composite_score"])
        flagged = df.filter(~pl.col("author_slug").is_in(list(control_set)))
        controls = df.filter(pl.col("author_slug").is_in(list(control_set)))
        fig = go.Figure()
        if flagged.height:
            fig.add_trace(
                go.Histogram(
                    x=flagged["composite_score"].to_list(),
                    name="flagged / above threshold",
                    opacity=0.75,
                )
            )
        if controls.height:
            fig.add_trace(
                go.Histogram(
                    x=controls["composite_score"].to_list(),
                    name="natural controls",
                    opacity=0.6,
                )
            )
        fig.update_layout(
            barmode="overlay",
            title="Composite AI-adoption score — newsroom survey",
            xaxis_title="Composite score",
            yaxis_title="Authors",
        )
        fig.show()

## Change-point timeline for flagged authors

One horizontal tick per author, at the earliest convergence-window start date. Clustered dates across multiple authors are the editorial-change signature §5c seeks to rule out.

In [ ]:
if payload is None or results_path is None:
    print("(no survey data — timeline skipped)")
else:
    import plotly.graph_objects as go  # noqa: I001

    run_dir = results_path.parent
    analysis_dir = ROOT / "data" / "analysis"
    rankings = payload.get("rankings", [])
    flagged_slugs = [
        r["author_slug"]
        for r in rankings
        if r.get("signal_strength") in {"weak", "moderate", "strong"}
    ]
    rows: list[dict[str, object]] = []
    for slug in flagged_slugs:
        candidate = analysis_dir / f"{slug}_result.json"
        if not candidate.is_file():
            continue
        data = json.loads(candidate.read_text(encoding="utf-8"))
        windows = data.get("convergence_windows") or []
        if not windows:
            continue
        earliest = min(w["start_date"] for w in windows)
        rows.append({"author": slug, "start_date": earliest})
    if not rows:
        print("(no flagged authors have convergence windows yet)")
    else:
        frame = pl.from_dicts(rows).sort("start_date")
        fig = go.Figure(
            go.Scatter(
                x=frame["start_date"].to_list(),
                y=frame["author"].to_list(),
                mode="markers",
                marker=dict(size=10, symbol="line-ns-open"),
            )
        )
        fig.update_layout(
            title="Earliest convergence-window start per flagged author",
            xaxis_title="Start date",
            yaxis_title="Author",
        )
        fig.show()

## Methodology & pre-registration status

Composite score weighting (from `forensics.survey.scoring.compute_composite_score`):

- With Pipeline C available: 25% Pipeline A (stylometric change-points) + 25% Pipeline B (embedding drift) + 15% Pipeline C (probability) + 35% convergence.
- Without Pipeline C: 30% A + 30% B + 40% convergence.

Signal tiers follow §1d: `STRONG` (composite ≥ 0.7 AND convergence score ≥ 0.5 AND max |d| ≥ 0.8 AND ≥ 2 windows), `MODERATE`, `WEAK`, `NONE`.

The next cell verifies whether the current analysis thresholds still match the pre-registered lock — a drift implies this result should be read as exploratory rather than confirmatory.

In [ ]:
from forensics.config import get_settings
from forensics.preregistration import verify_preregistration

try:
    verification = verify_preregistration(get_settings())
    print(f"Pre-registration status: {verification.status}")
    print(verification.message)
except Exception as exc:  # pragma: no cover - presentation-only
    print(f"Pre-registration check skipped: {exc}")